In [111]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt 

np.random.seed(42)
n_winners = 10
n_losers = 10
n_noise = 80
n_months = 60 # 5 years 
monthly_vol = 0.08

## Synthetic Data Generation

In [50]:
winners = np.random.normal(loc=0.02, scale=monthly_vol, size=(n_months, n_winners))
losers = np.random.normal(loc=-0.02, scale=monthly_vol, size=(n_months, n_losers))
noise = np.random.normal(loc=0, scale=monthly_vol, size=(n_months, n_noise))

In [40]:
# Sanity Check

print(f"winners mean - {winners.mean()}")
for i, w in enumerate(winners):
    print(f'Winner: {i} - avg return: {w.mean():.4f}')

print(f"losers mean - {losers.mean()}")
for i, l in enumerate(losers):
    print(f'Loser: {i} - avg return: {l.mean():.4f}')

print(f"noise mean - {noise.mean()}")
for i, n in enumerate(noise):
    print(f'Noise: {i} - avg return: {n.mean():.4f}')
    if i == 40:
        break

winners mean - 0.025182300378369035
Winner: 0 - avg return: 0.0288
Winner: 1 - avg return: 0.0239
Winner: 2 - avg return: 0.0230
Winner: 3 - avg return: 0.0213
Winner: 4 - avg return: 0.0168
Winner: 5 - avg return: 0.0231
Winner: 6 - avg return: 0.0248
Winner: 7 - avg return: 0.0350
Winner: 8 - avg return: 0.0384
Winner: 9 - avg return: 0.0165
losers mean - -0.019139986124284435
Loser: 0 - avg return: -0.0274
Loser: 1 - avg return: -0.0192
Loser: 2 - avg return: -0.0090
Loser: 3 - avg return: -0.0151
Loser: 4 - avg return: -0.0116
Loser: 5 - avg return: -0.0088
Loser: 6 - avg return: -0.0255
Loser: 7 - avg return: -0.0408
Loser: 8 - avg return: -0.0234
Loser: 9 - avg return: -0.0106
noise mean - -0.0006340463494431037
Noise: 0 - avg return: -0.0056
Noise: 1 - avg return: -0.0024
Noise: 2 - avg return: 0.0220
Noise: 3 - avg return: 0.0023
Noise: 4 - avg return: -0.0048
Noise: 5 - avg return: 0.0111
Noise: 6 - avg return: -0.0043
Noise: 7 - avg return: 0.0068
Noise: 8 - avg return: -0.00

In [71]:
universe = np.hstack([winners, losers, noise])
df = pl.DataFrame(universe)
names = [f"w{i}" for i in range(n_winners)] + [f"l{i}" for i in range(n_losers)] + [f"n{i}" for i in range(n_noise)]

In [78]:
df = df.rename({f"column_{i}": name for i, name in enumerate(names)})

## WML Portfolio Constrauction

12-1 momentum sort on the synthetic data

Steps:
1. Compute the formation period return - For each stock at each month t, compute the cumulative return from t-12 to t-2. This is 11 months of data, skipping the most recent month. Write a comment explaining why we skip t-1 (short-term reversal)
2. Rank stocks into deciles Each month, rank all stocks by their formation return. Assign o deciles 1 (lowest) through 10 (highest)
3. Compute decile returns - For each decile, compute the equal weighted return in the following month (the holding period). In the real data we will use value-weighting - equal weighting is fine for synthetic verficiation
4. Construct VML - WML return = decile 10 return - decile 1 return. This is the monthly time series

Gate - Four checks, all must pass
- Decile 10 (winners) has higher mean return than Decile 1 (loseers)
- WML mean monthly return is positive
- WML skewness is negative (crashes are left-tail events - positive skewness means construction is inverted)
- Print the top 5 worst WML months and eyeball whether they cluster in periods when losers out perform winners - they should

In [ ]:
# Compute formation returns on each stock
formation = df.select(
    pl.col(col)
    .shift(2)
    .rolling_map(lambda x: (1 + x).product() - 1, window_size=11)
    .alias('{}_formation'.format(col))

    for col in df.columns
)

In [160]:
# Validate shape
formation.shape

(60, 100)

In [164]:
# Sanity check to see formation was successfull and winner and loser means are evident
winner_cols = [f'w{i}_formation' for i in range(10)]
loser_cols = [f'l{i}_formation' for i in range(10)]

winner_mean = formation.select(winner_cols).unpivot().select(pl.col('value').mean())[0,0]
loser_mean = formation.select(loser_cols).unpivot().select(pl.col('value').mean())[0,0]

print(f"Winner mean formation return: {winner_mean:.4f}")
print(f"Loser mean formation return: {loser_mean:.4f}")

Winner mean formation return: 0.2505
Loser mean formation return: -0.1268


In [ ]:
# Convert formation data into long format with month as index
formation_long = formation.with_row_index('month').unpivot(
    index='month',
    variable_name='ticker',
    value_name='formation_return'
).drop_nulls()

In [ ]:
# Fix ticker names and compute ranks grouped by month
formation_long = formation_long.with_columns(
    pl.col('ticker').str.replace('_formation', ''),
    pl.col('formation_return').rank()
    .over('month').alias('rank')
)

In [ ]:
# Turn ranks into deciles using ceil and / 10
formation_long = formation_long.with_columns(
    (pl.col('rank') / 10).ceil().cast(pl.Int32).alias('decile')
)

In [ ]:
# Convert returns into the same month index format
returns_long = df.with_row_index('month').unpivot(
    index='month',
    variable_name='ticker',
    value_name='return'
)

In [188]:
# Join our formation calculations onto the long returns df. Joined on ticker and used inner join as we dont require the null months
joined = returns_long.join(formation_long, on=['month', 'ticker'], how='inner').drop('rank')

In [ ]:
# Calculate decile returns
decile_returns = joined.group_by(['month', 'decile']).agg([
    pl.col('return').mean().alias('decile_return')
]).sort(['month', 'decile'])

In [196]:
# Construct WML portfolio by taking the difference between decile 10 and decile 1
wml = decile_returns.filter(pl.col('decile') == 10).join(
    decile_returns.filter(pl.col('decile') == 1),
    on='month',
    suffix='_loser'
).with_columns(
    (pl.col('decile_return') - pl.col('decile_return_loser')).alias('wml_return')
).select(['month', 'wml_return'])

In [198]:
# Gate checks
import scipy.stats as stats

wml_vals = wml['wml_return'].to_numpy()

mean = wml_vals.mean()
sharpe = (mean / wml_vals.std()) * np.sqrt(12)
skewness = stats.skew(wml_vals)

print(f"Mean monthly return: {mean:.4f}")
print(f"Annualised Sharpe: {sharpe:.4f}")
print(f"Skewness: {skewness:.4f}")

# Worst 5 months
print("\nWorst 5 WML months:")
print(wml.sort('wml_return').head(5))

Mean monthly return: 0.0087
Annualised Sharpe: 0.8549
Skewness: -0.2901

Worst 5 WML months:
shape: (5, 2)
┌───────┬────────────┐
│ month ┆ wml_return │
│ ---   ┆ ---        │
│ u32   ┆ f64        │
╞═══════╪════════════╡
│ 12    ┆ -0.08624   │
│ 56    ┆ -0.059312  │
│ 26    ┆ -0.055181  │
│ 33    ┆ -0.046476  │
│ 47    ┆ -0.040434  │
└───────┴────────────┘


### Summary:

In this section, we built the WML portfolio, using the desired frameworked cited in the material

We ensured to follow the t-12 - t-1 framework for calculating cumulative returns, aswell as the decile ranking logic, this remained intact throughout and validated.

Results: We can see:
- Mean Monthly Return: 0.087
- Sharpe: 0.8549
- Skewness: -0.2901

This validates the architecture and what found in the paper, and this analysis reflect this accuracy as the skewness shows crashes are left-skewed



## Bear Market Indicator

The panic state flag and verficiation that bad WML months cluster in panic states.

For synthetic data you need to also generate a synthetic market return series. Keep it simple - a random walk with occasional large negative periods that you manually insert to create known bear states.

1. Compute the rolling 24-month market return - At each month t, sum the prior 24 monthly market returns. If negative, I_B = 1.
2. Identify the panic state - Panic = I_B is 1 AND current  month market return is positive. This is the bear state + sudden rally combination
3. Conditional WML returns - Compute mean WML return when: (a) normal state, (b) bear state, (c) panic state. The ordering should be: normal >> bear > panic.
4. Regression - Run the regression 
WML_t = α₀ + α_B * I_B + β₀ * R_m + β_B * I_B * R_m + ε

On synthetic data the coefficients won't be meaninful - but the regression should run without errors, the signs should be directionally correct, and you should understand every term.

Gate:
- Mean WML return in panic states lower than in normal states
- Regression runs and coefficients have correct signs (α_B negative, β_B negative)
- You can explain in plan English what each regression coefficient means

In [ ]:
# Generate synthetic market returns data for more general view
np.random.seed(99)
market_returns = np.random.normal(loc=0.005, scale=0.04, size=60)
market_returns = pl.Series('market_return', market_returns)

In [ ]:
# Insert some volatility crashes into the market returns
market_arr = market_returns.to_numpy().copy()
market_arr[20:38] = np.random.normal(loc=-0.03, scale=0.04, size=18)
market_returns = pl.Series('market_return', market_arr)

In [218]:
# Apply rolling 24m cumulative return to market returns to identify crashes
market_returns = pl.DataFrame(market_returns).with_columns(
    (pl.col('market_return')
    .rolling_map(lambda x: (1 + x).product() - 1, window_size=24)).alias('rolling_24m')
)

In [221]:
# Create crash indicator based on rolling 24m cumulative return being negative
market_df = market_returns.with_columns(
    pl.when(pl.col('rolling_24m') < 0)
    .then(pl.lit(1))
    .otherwise(pl.lit(0)).alias('bear_state')
)

In [ ]:
# Add month index to market_df for joining
market_df = market_df.with_row_index('month')

In [ ]:
# Join WML returns with market state and returns
wml_with_state = wml.join(market_df.select(['month', 'bear_state', 'market_return']), on='month')

In [232]:
print(wml_with_state.group_by('bear_state').agg(
    pl.col('wml_return').mean().alias('mean_wml')
).sort('bear_state'))

shape: (2, 2)
┌────────────┬───────────┐
│ bear_state ┆ mean_wml  │
│ ---        ┆ ---       │
│ i32        ┆ f64       │
╞════════════╪═══════════╡
│ 0          ┆ -0.000586 │
│ 1          ┆ 0.012497  │
└────────────┴───────────┘


In [236]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [238]:
X = wml_with_state.select(['bear_state', 'market_return']).to_numpy()

# Add interaction term and constant
interaction = (X[:, 0] * X[:, 1]).reshape(-1, 1)
const = np.ones((len(X), 1))
X_full = np.hstack([const, X, interaction])

y = wml_with_state['wml_return'].to_numpy()

# OLS: beta = (X'X)^-1 X'y
beta = np.linalg.lstsq(X_full, y, rcond=None)[0]

print(f"Constant:        {beta[0]:.4f}")
print(f"Bear state:      {beta[1]:.4f}")
print(f"Market return:   {beta[2]:.4f}")
print(f"Bear x Market:   {beta[3]:.4f}")

Constant:        -0.0004
Bear state:      0.0166
Market return:   -0.1374
Bear x Market:   0.4729


Results:

Constant - The constant tells us that we start with an initial -0.0004 return when the market is Bear (0) and return is 0

Bear State - This 0.0166 explains the additional WML return when you're in a bear state, holding market return as constant.

Market return - For every 1% the market gains, WML loses 0.14%. This is the market beta of the momentum portfolio - negative because the short leg (losers) has higher market beta than the long leg (winners). This one has the right sign

Bear x Market - The extra sensitivity of WML to market returns specifically during bear states. 

## Variance Forecasting

Two variance forecasters - realised volatility and GJR-GARCH - then combining them

The paper shows each contributes independetly. Realised vol captures the recent observable level. GJR-GARCH captures the persistence and asymmetry of volatility. Together they predict better than either alone.

1. Realised Volatility - Convert monthly return to approximate daily returns (divide by 21). Compute the standard deviation of the past 126 daily returns. This is your realised vol estimated σ_realised

2. GJR-GARCH - This is the most technically complex part of the paper, Standard GARCH says: Tomorrow's variance is a weighted average of yesterday'svariance and yesterday's squared return shock.
GJR-CARCH adds one term: if yesterday's shock was negative, add an extra penalty y on top. Bad days increase future variance more than good days of the same size. This asymmetry is realistic and important for momentum specifically because crashes product large negative shocks.

σ²_t = ω + β·σ²_{t-1} + α·ε²_{t-1} + γ·I(ε_{t-1} < 0)·ε²_{t-1}

Params:
ω (long-run variance floor), β (persistence), α (ARCH effect), γ (asymmetry penalty for negative shocks). For the synthetic notebook, use the arch Python library:

3. Combine the two forecastst - run a simple OLS regression of next-month realised variance on both forecasters: σ²_realised_{t+1} = a + b₁·σ²_GARCH_t + b₂·σ²_realised_t + ε

The fitted values from this regression are your combined variance forecast.

4. Validate - Plot forecast variance over time. Manually insert a volatility spike into your data and verify the forecast responds to it. The response should be fast (GARCH catches it quickly) and persistnet (it doesn't immediately return to normal)

Gates:
- GJR-GARCH fits without errors
- Asymmetry parameter γ is positive (negative shocks increase future variance more — if it's negative something is wrong)
- Variance forecast spikes when you insert a bad period into synthetic data
- Combined forecast is better than either component alone (lower MSE against next-period realised variance)


In [249]:
# Calcualte realised volatility from WML returns
wml_with_vol = wml.with_columns(
    pl.col('wml_return')
    .rolling_std(window_size=6)
    .alias('realised_vol')
)

In [241]:
# Fitting arch model to WML to model GJR-GARCH effects
from arch import arch_model

wml_returns = wml['wml_return'].to_numpy() * 100

model = arch_model(wml_returns, p=1, o=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

                   Constant Mean - GJR-GARCH Model Results                    
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -128.031
Distribution:                  Normal   AIC:                           266.061
Method:            Maximum Likelihood   BIC:                           275.417
                                        No. Observations:                   48
Date:                Wed, Apr 22 2026   Df Residuals:                       47
Time:                        19:37:51   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu             0.8982      0.505      1.777  7.551e-02 [-9.22

In [242]:
# GJR-GARCH asymmetry (gamma) is effectively zero on synthetic data.
# This is a sample size issue — 48 monthly observations is insufficient
# for GARCH to identify volatility clustering and asymmetry.
# On real daily data (2010-2026, ~3,500+ observations) this will work correctly.
# Gate check deferred to Notebook 2.

Results

omega - This represents the baseline volatility of the model being 2.3760, this reflects that the variance will never drop below this level

alpha - This determines how much the previous month effects the current months variance, it will be effected by 2.0331e-12 if the previous month has extreme returns

gamma - This variable is used to add extra volatility after negative days, terms of our model, this adds more volatility to the forecast if the previous days were negatively volatile. This captures the fact that a market is more volatile after falls then after rises.

beta - This reflects the life span of the volatility, in this case, 0.7844 tells us that volatility doesn't decay over night, it will slowly decay over time

In [251]:
wml_forecast = wml_with_state.join(wml_with_vol.select(['month', 'realised_vol']), on='month')

In [252]:
wml_forecast.shape

(48, 5)

In [255]:
wml_forecast = wml_forecast.drop_nulls()

In [ ]:
# Create interation model 
X = wml_forecast.select(['bear_state', 'realised_vol']).to_numpy()

interaction = (X[:, 0] * X[:, 1]).reshape(-1, 1)
const = np.ones((len(X), 1))
X_full = np.hstack([const, X, interaction])

y = wml_forecast['wml_return'].to_numpy()

beta = np.linalg.lstsq(X_full, y, rcond=None)[0]

print(f"Constant (γ₀):        {beta[0]:.4f}")
print(f"Bear state (γ_B):     {beta[1]:.4f}")
print(f"Realised vol (γ_σ²):  {beta[2]:.4f}")
print(f"Interaction (γ_int):  {beta[3]:.4f}")

Constant (γ₀):        0.0565
Bear state (γ_B):     -0.0082
Realised vol (γ_σ²):  -1.4694
Interaction (γ_int):  0.3780


In [ ]:
# Fitted values - these are the forecasts 
mu_forecast = X_full @ beta

# Add back to DataFrame and shift towards one month (no lookahead)
wml_forecast = wml_forecast.with_columns(
    pl.Series('mu_forecast', mu_forecast).shift(1)
)



In [ ]:
# Drop nulls before weight calculation
wml_forecast = wml_forecast.drop_nulls()

In [ ]:
# Define weight calculations
lam = 1
w = ((1/(2*lam)) * (wml_forecast['mu_forecast']/wml_forecast['realised_vol'] ** 2)).clip(lower_bound=0)

In [ ]:
# Add column with label weight from weight calculations
wml_forecast = wml_forecast.with_columns(
    w.alias('weight')
)

In [ ]:
# Add return values based on weighting and realised volatiltiy
wml_forecast = wml_forecast.with_columns([
    pl.col('wml_return').alias('static'),
    (pl.col('wml_return') * pl.col('weight')).alias('dynamic'),
    (pl.col('wml_return') / pl.col('realised_vol')).alias('cvol')
])

In [276]:
# Optimise lambda
from scipy.optimize import minimize_scalar

target_vol = wml_forecast['static'].std() * np.sqrt(12)

cvol_raw = wml_forecast['wml_return'] / wml_forecast['realised_vol']
cvol_scaled = cvol_raw * (target_vol / (cvol_raw.std() * np.sqrt(12)))

wml_forecast = wml_forecast.with_columns(
    cvol_scaled.alias('cvol')
)

def vol_diff(lam):
    w = (1 / (2 * lam)) * (wml_forecast['mu_forecast'] / wml_forecast['realised_vol'] ** 2)
    w = w.clip(lower_bound=0)
    dynamic_returns = wml_forecast['wml_return'] * w
    return (dynamic_returns.std() * np.sqrt(12) - target_vol) ** 2

result = minimize_scalar(vol_diff, bounds=(0.01, 100), method='bounded')
lambda_opt = result.x
print(f"Optimal lambda: {lambda_opt:.4f}")

Optimal lambda: 9.8830


In [277]:
# Run new weight calculation with optimal lambda
w_calibrated = ((1 / (2 * lambda_opt)) * 
    (wml_forecast['mu_forecast'] / wml_forecast['realised_vol'] ** 2)
).clip(lower_bound=0)

# Apply forecast to our existing dataframe
wml_forecast = wml_forecast.with_columns(
    (pl.col('wml_return') * pl.Series(w_calibrated)).alias('dynamic_calibrated')
)

# Run through each strategy and calculate gate check metrics
for strategy in ['static', 'dynamic_calibrated', 'cvol']:
    ret = wml_forecast[strategy].to_numpy()
    sharpe = (ret.mean() / ret.std()) * np.sqrt(12)
    skew = stats.skew(ret)
    worst = ret.min()
    print(f"{strategy:20s} Sharpe: {sharpe:.3f}  Skew: {skew:.3f}  Worst month: {worst:.3f}")

static               Sharpe: 1.105  Skew: -0.069  Worst month: -0.059
dynamic_calibrated   Sharpe: 1.210  Skew: 1.663  Worst month: -0.057
cvol                 Sharpe: 1.389  Skew: 0.032  Worst month: -0.055


In [278]:
# Dynamic skewness is positive on synthetic data — expected.
# The mean forecast (μ) is too noisy on 42 observations to reliably 
# identify crash periods. On real data the skewness should be less 
# negative than static but not strongly positive.